In [ ]:
# ============================================================
# IMPORTS ET CHARGEMENT DES DATASETS

import os 
import sys
import pandas as pd
sys.path.append(os.path.abspath("..")) 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error
from src.models.train_model_features import preparer_X_y, creer_modele_XGB, calculer_mae_cv, tester_apport_features, tester_groupes_features


dvf_nettoye = pd.read_csv("../data/processed/dvf_nettoye.csv", dtype={"code_commune": "string"})
X, y = preparer_X_y(dvf_nettoye)

print(X.shape)
print(y.shape)
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    # stratify=X["type_local"]
)

mae_reference, std_reference = calculer_mae_cv(
    X_train,
    y_train
)

print("MAE CV :", round(mae_reference, 2))
print("Écart-type :", round(std_reference, 2))

resultats_features = tester_apport_features(
    X_train,
    y_train
)

display(resultats_features.round(2))

(110889, 13)
(110889,)
['lot1_surface_carrez', 'nombre_lots', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'surface_terrain', 'longitude', 'latitude', 'annee_mutation', 'mois_mutation', 'surface_par_piece', 'log_surface', 'distance_lyon_km']
MAE CV : 679.47
Écart-type : 4.01
MAE avec toutes les features : 679.47 €/m²
Test sans : lot1_surface_carrez
Test sans : nombre_lots
Test sans : type_local
Test sans : surface_reelle_bati
Test sans : nombre_pieces_principales
Test sans : surface_terrain
Test sans : longitude
Test sans : latitude
Test sans : annee_mutation
Test sans : mois_mutation
Test sans : surface_par_piece
Test sans : log_surface
Test sans : distance_lyon_km


,Feature retirée,MAE CV,Écart-type,Différence MAE
6,longitude,725.37,4.74,45.90
7,latitude,720.92,4.90,41.45
8,annee_mutation,702.78,3.99,23.31
5,surface_terrain,695.20,3.91,15.73
12,distance_lyon_km,689.79,4.62,10.33
1,nombre_lots,689.38,4.79,9.91
0,lot1_surface_carrez,688.80,4.10,9.33
2,type_local,682.12,4.13,2.66
9,mois_mutation,680.55,3.75,1.08
11,log_surface,679.96,4.70,0.49


In [2]:
features_a_supprimer = ["code_commune"]

X_train_selection = X_train.drop(
    columns=features_a_supprimer,
    errors="ignore"
)

mae_selection, std_selection = calculer_mae_cv(
    X_train_selection,
    y_train
)

print("MAE avec toutes les features :", round(683.35, 2))
print("MAE sans code_commune :", round(mae_selection, 2))
print("Écart-type :", round(std_selection, 2))

MAE avec toutes les features : 683.35
MAE sans code_commune : 679.47
Écart-type : 4.01


In [3]:
groupes_a_tester = {
    "sans_code_commune": ["code_commune"],

    "sans_log_surface": ["log_surface"],

    "sans_surface_par_piece": ["surface_par_piece"],

    "sans_features_surface_creees": [
        "log_surface",
        "surface_par_piece"
    ],

    "sans_coordonnees": [
        "latitude",
        "longitude",
        "distance_lyon_km"
    ]
}

resultats_groupes = tester_groupes_features(
    X_train,
    y_train,
    groupes_a_tester
)

display(resultats_groupes.round(2))

,Configuration,MAE CV,Écart-type,Différence MAE
0,sans_code_commune,679.47,4.01,0.00
2,sans_surface_par_piece,679.84,4.24,0.37
1,sans_log_surface,679.96,4.70,0.49
3,sans_features_surface_creees,680.71,4.29,1.24
4,sans_coordonnees,1048.33,2.56,368.86
